In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision.transforms import v2
from torchvision.io import decode_image
from torch.utils.data import Dataset
import os
import zipfile

In [3]:
import zipfile
with zipfile.ZipFile("/content/drive/MyDrive/hanyang/SIR2.zip") as z:
    z.extractall("/content/data/SIR2")


In [ ]:

for f in os.listdir("/content/data/SIR2"):
    p = os.path.join("/content/data/SIR2", f)
    if os.path.isdir(p):
        print(f"[DIR]  {f}")
    else:
        print(f"[FILE] {f}  ({os.path.getsize(p)/1e6:.1f} Mo)")

[FILE] Wildscene.zip  (5.9 Mo)
[FILE] Postcard Dataset.zip  (177.7 Mo)
[FILE] SolidObjectDataset.zip  (13.8 Mo)


In [ ]:
base = "/content/data/SIR2"
inner = ["SolidObjectDataset.zip", "Postcard Dataset.zip", "Wildscene.zip"]

for z in inner:
    with zipfile.ZipFile(os.path.join(base, z)) as zf:
        names = zf.namelist()
        print(f"\n=== {z} : {len(names)} entrées ===")
        for n in names[:25]:
            print("  ", n)


=== SolidObjectDataset.zip : 862 entrées ===
   SolidObjectDataset/
   SolidObjectDataset/1/
   SolidObjectDataset/1/Focus/
   SolidObjectDataset/1/Focus/11/
   SolidObjectDataset/1/Focus/11/g.jpg
   SolidObjectDataset/1/Focus/11/m.jpg
   SolidObjectDataset/1/Focus/11/r.jpg
   SolidObjectDataset/1/Focus/13/
   SolidObjectDataset/1/Focus/13/g.jpg
   SolidObjectDataset/1/Focus/13/m.jpg
   SolidObjectDataset/1/Focus/13/r.jpg
   SolidObjectDataset/1/Focus/16/
   SolidObjectDataset/1/Focus/16/g.jpg
   SolidObjectDataset/1/Focus/16/m.jpg
   SolidObjectDataset/1/Focus/16/r.jpg
   SolidObjectDataset/1/Focus/19/
   SolidObjectDataset/1/Focus/19/g.jpg
   SolidObjectDataset/1/Focus/19/m.jpg
   SolidObjectDataset/1/Focus/19/r.jpg
   SolidObjectDataset/1/Focus/22/
   SolidObjectDataset/1/Focus/22/g.jpg
   SolidObjectDataset/1/Focus/22/m.jpg
   SolidObjectDataset/1/Focus/22/r.jpg
   SolidObjectDataset/1/Focus/27/
   SolidObjectDataset/1/Focus/27/g.jpg

=== Postcard Dataset.zip : 840 entrées ===
   

In [ ]:
base = "/content/data/SIR2"
inner = {
    "SolidObjectDataset.zip": "SolidObject",
    "Postcard Dataset.zip": "Postcard",
    "Wildscene.zip": "Wild",
}
for z, out in inner.items():
    with zipfile.ZipFile(os.path.join(base, z)) as zf:
        zf.extractall(os.path.join(base, out))
    print(f"{z} → {out}/ OK")

SolidObjectDataset.zip → SolidObject/ OK
Postcard Dataset.zip → Postcard/ OK
Wildscene.zip → Wild/ OK


In [16]:
for dirpath, dirnames, filenames in os.walk(base):
    if len(filenames)>=3:
        print(filenames)

['Wildscene.zip', 'Postcard Dataset.zip', 'SolidObjectDataset.zip']
['eb-10-m-3.png', 'eb-10-r-3.png', 'eb-10-g-3.png']
['eb-5-r-32.png', 'eb-5-g-32.png', 'eb-5-m-32.png']
['eb-3-r-32.png', 'eb-3-m-32.png', 'eb-3-g-32.png']
['hi-10-g-3.png', 'hi-10-m-3.png', 'hi-10-r-3.png']
['hi-5-g-32.png', 'hi-5-r-32.png', 'hi-5-m-32.png']
['hi-3-r-32.png', 'hi-3-g-32.png', 'hi-3-m-32.png']
['ei-10-m-3.png', 'ei-10-r-3.png', 'ei-10-g-3.png']
['ei-5-g-32.png', 'ei-5-r-32.png', 'ei-5-m-32.png']
['ei-3-m-32.png', 'ei-3-g-32.png', 'ei-3-r-32.png']
['hb-10-g-3.png', 'hb-10-r-3.png', 'hb-10-m-3.png']
['hb-5-r-32.png', 'hb-5-m-32.png', 'hb-5-g-32.png']
['hb-3-m-32rs1.jpg', 'hb-3-r-32rs1.jpg', 'hb-3-g-32rs1.jpg']
['ea-10-m-3.png', 'ea-10-r-3.png', 'ea-10-g-3.png']
['ea-5-m-32.png', 'ea-5-r-32.png', 'ea-5-g-32.png']
['ea-3-g-32.png', 'ea-3-r-32.png', 'ea-3-m-32.png']
['ie-10-g-3.png', 'ie-10-m-3.png', 'ie-10-r-3.png']
['ie-5-r-32.png', 'ie-5-m-32.png', 'ie-5-g-32.png']
['ie-3-m-32.png', 'ie-3-r-32.png', 'ie-

In [10]:
def det_role(str):
    if "r" in str:
        return "r"
    elif "m" in str:
        return "m"
    else:
        return "g"

In [21]:
def build_index(subset_root):
    """
    input: str
    output: list[dict]
    """
    dico = []
    for dirpath, dirnames, filenames in os.walk(subset_root):
        if len(filenames)==3:
            if "jpg" in filenames[0][-3:] or "png" in filenames[0][-3]:
                dico.append({det_role(i): os.path.join(dirpath, i) for i in filenames})

    return dico


# index = build_index(base)
# print(index[15])


In [ ]:
class SirDataset(Dataset):
    """
    Dataset SIR
    """
    def __init__(self, base):
        self.index = build_index(base)

    def __len__(self):
        return len(self.index)

